# Neural Network Fundamentals: From Perceptron to Deep Learning

This notebook covers:
1. **Perceptron to Multi-Layer Networks**: Building blocks of neural networks
2. **Activation Functions**: ReLU, Sigmoid, Tanh and when to use each
3. **Loss Functions**: MSE vs Cross-Entropy
4. **Forward Pass in PyTorch**: Implementing networks with nn.Module

By the end, you'll build and train a complete neural network from scratch!

## Setup and Installation

In [ ]:
# Install PyTorch if needed
# !pip install torch torchvision matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print("Setup complete!")

---
# Part 1: From Perceptron to Multi-Layer Networks

## 1.1 The Perceptron: Simplest Neural Unit

In [ ]:
class Perceptron:
    """
    Simple perceptron: weighted sum + activation.

    Formula: y = activation(w·x + b)
    """
    def __init__(self, input_size):
        self.weights = torch.randn(input_size, requires_grad=True)
        self.bias = torch.randn(1, requires_grad=True)

    def forward(self, x):
        # Weighted sum
        z = torch.dot(x, self.weights) + self.bias
        # Step activation (1 if z > 0, else 0)
        y = (z > 0).float()
        return y

# Test perceptron
perceptron = Perceptron(input_size=3)
x = torch.tensor([1.0, 2.0, 3.0])
output = perceptron.forward(x)

print("Perceptron Test:")
print(f"  Input: {x}")
print(f"  Weights: {perceptron.weights}")
print(f"  Bias: {perceptron.bias}")
print(f"  Output: {output}")

## 1.2 Visualizing Perceptron Decision Boundary

In [ ]:
# Create 2D data (for visualization)
np.random.seed(42)
X_class0 = np.random.randn(50, 2) + np.array([2, 2])
X_class1 = np.random.randn(50, 2) + np.array([-2, -2])
X = np.vstack([X_class0, X_class1])
y = np.hstack([np.zeros(50), np.ones(50)])

# Plot data
plt.figure(figsize=(10, 6))
plt.scatter(X[y==0][:, 0], X[y==0][:, 1], c='blue', label='Class 0', s=50, alpha=0.7)
plt.scatter(X[y==1][:, 0], X[y==1][:, 1], c='red', label='Class 1', s=50, alpha=0.7)

# Perceptron decision boundary (random initialization)
w = np.array([1.0, 1.0])
b = 0
x_line = np.linspace(-5, 5, 100)
y_line = -(w[0] * x_line + b) / w[1]
plt.plot(x_line, y_line, 'g--', linewidth=2, label='Decision Boundary')

plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Perceptron: Linear Decision Boundary')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Perceptron can only create LINEAR decision boundaries!")

## 1.3 The XOR Problem: Perceptron's Limitation

In [ ]:
# XOR data (NOT linearly separable!)
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])  # XOR labels

plt.figure(figsize=(8, 6))
plt.scatter(X_xor[y_xor==0][:, 0], X_xor[y_xor==0][:, 1],
           c='blue', s=200, label='Class 0', marker='o')
plt.scatter(X_xor[y_xor==1][:, 0], X_xor[y_xor==1][:, 1],
           c='red', s=200, label='Class 1', marker='s')

plt.xlabel('Input 1')
plt.ylabel('Input 2')
plt.title('XOR Problem: No Single Line Can Separate These Classes!', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(-0.5, 1.5)
plt.ylim(-0.5, 1.5)
plt.show()

print("❌ Single perceptron CANNOT solve XOR")
print("✅ Multi-layer network CAN solve XOR!")

## 1.4 Building a Multi-Layer Network

In [ ]:
class MultiLayerNetwork:
    """
    Manual 3-layer network to understand the math.

    Architecture: Input → Hidden1 → Hidden2 → Output
    """
    def __init__(self, input_size, hidden1, hidden2, output_size):
        # Layer 1
        self.W1 = torch.randn(input_size, hidden1, requires_grad=True)
        self.b1 = torch.randn(hidden1, requires_grad=True)

        # Layer 2
        self.W2 = torch.randn(hidden1, hidden2, requires_grad=True)
        self.b2 = torch.randn(hidden2, requires_grad=True)

        # Layer 3
        self.W3 = torch.randn(hidden2, output_size, requires_grad=True)
        self.b3 = torch.randn(output_size, requires_grad=True)

    def forward(self, x):
        # Layer 1: Linear → ReLU
        z1 = torch.matmul(x, self.W1) + self.b1
        a1 = torch.relu(z1)

        # Layer 2: Linear → ReLU
        z2 = torch.matmul(a1, self.W2) + self.b2
        a2 = torch.relu(z2)

        # Layer 3: Linear (output)
        z3 = torch.matmul(a2, self.W3) + self.b3

        return z3

# Create network: 4 inputs → 8 → 4 → 2 outputs
net = MultiLayerNetwork(input_size=4, hidden1=8, hidden2=4, output_size=2)

# Test forward pass
x_test = torch.randn(5, 4)  # Batch of 5 samples
output = net.forward(x_test)

print("Multi-Layer Network Test:")
print(f"  Input shape: {x_test.shape}")
print(f"  Output shape: {output.shape}")
print(f"\nLayer 1 weights shape: {net.W1.shape}")
print(f"Layer 2 weights shape: {net.W2.shape}")
print(f"Layer 3 weights shape: {net.W3.shape}")

### 🔍 Exercise 1.1: Count Parameters

Calculate the total number of parameters in the network above (weights + biases).

In [ ]:
# TODO: Calculate total parameters
# Formula: For each layer: (in × out) + out

layer1_params = None  # Calculate
layer2_params = None  # Calculate
layer3_params = None  # Calculate
total_params = None   # Sum

print(f"Layer 1 parameters: {layer1_params}")
print(f"Layer 2 parameters: {layer2_params}")
print(f"Layer 3 parameters: {layer3_params}")
print(f"Total parameters: {total_params}")

# Verify with actual count
actual_total = net.W1.numel() + net.b1.numel() + net.W2.numel() + net.b2.numel() + net.W3.numel() + net.b3.numel()
print(f"\nActual total: {actual_total}")

---
# Part 2: Activation Functions

## 2.1 Why Activation Functions Matter

In [ ]:
# Demonstrate: Without activation, layers collapse

def two_layers_no_activation(x, W1, b1, W2, b2):
    """Two linear layers WITHOUT activation."""
    z1 = torch.matmul(x, W1) + b1
    z2 = torch.matmul(z1, W2) + b2
    return z2

# Random weights
x = torch.tensor([[1.0, 2.0]])
W1 = torch.tensor([[0.5, 0.3], [0.2, 0.4]])
b1 = torch.tensor([0.1, 0.2])
W2 = torch.tensor([[0.6], [0.7]])
b2 = torch.tensor([0.3])

# Method 1: Two layers
output_two = two_layers_no_activation(x, W1, b1, W2, b2)

# Method 2: Collapsed to single layer
W_combined = torch.matmul(W1, W2)
b_combined = torch.matmul(b1, W2) + b2
output_single = torch.matmul(x, W_combined) + b_combined

print("Without Activation Functions:")
print(f"  Two layers output: {output_two}")
print(f"  Single layer output: {output_single}")
print(f"  Are they equal? {torch.allclose(output_two, output_single)}")
print("\n⚠️ Depth provides NO benefit without activation!")

## 2.2 ReLU (Rectified Linear Unit)

In [ ]:
# ReLU: max(0, x)
x = torch.linspace(-3, 3, 100)
y_relu = F.relu(x)

plt.figure(figsize=(10, 6))
plt.plot(x.numpy(), y_relu.numpy(), linewidth=2, label='ReLU', color='blue')
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.grid(True, alpha=0.3)
plt.xlabel('Input (x)', fontsize=12)
plt.ylabel('Output', fontsize=12)
plt.title('ReLU Activation: max(0, x)', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.show()

# Properties
print("ReLU Properties:")
print("  Output range: [0, ∞)")
print("  Derivative: 1 if x > 0, else 0")
print("  Use case: Hidden layers (default choice)")
print("  Advantage: Fast, no vanishing gradient")
print("  Disadvantage: 'Dying neurons' if stuck at 0")

## 2.3 Sigmoid

In [ ]:
# Sigmoid: 1 / (1 + e^(-x))
y_sigmoid = torch.sigmoid(x)

plt.figure(figsize=(10, 6))
plt.plot(x.numpy(), y_sigmoid.numpy(), linewidth=2, label='Sigmoid', color='green')
plt.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Midpoint (0.5)')
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.grid(True, alpha=0.3)
plt.xlabel('Input (x)', fontsize=12)
plt.ylabel('Output', fontsize=12)
plt.title('Sigmoid Activation: 1 / (1 + exp(-x))', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.ylim(-0.1, 1.1)
plt.show()

print("Sigmoid Properties:")
print("  Output range: (0, 1)")
print("  Use case: Binary classification OUTPUT layer")
print("  Advantage: Outputs as probabilities")
print("  Disadvantage: Vanishing gradient in deep networks")

## 2.4 Tanh (Hyperbolic Tangent)

In [ ]:
# Tanh
y_tanh = torch.tanh(x)

plt.figure(figsize=(10, 6))
plt.plot(x.numpy(), y_tanh.numpy(), linewidth=2, label='Tanh', color='orange')
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.grid(True, alpha=0.3)
plt.xlabel('Input (x)', fontsize=12)
plt.ylabel('Output', fontsize=12)
plt.title('Tanh Activation', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.ylim(-1.2, 1.2)
plt.show()

print("Tanh Properties:")
print("  Output range: (-1, 1)")
print("  Use case: Zero-centered outputs, RNN gates")
print("  Advantage: Zero-centered (better than sigmoid)")
print("  Disadvantage: Still has vanishing gradient")

## 2.5 Comparison: All Three Activations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ReLU
axes[0].plot(x.numpy(), y_relu.numpy(), linewidth=2, color='blue')
axes[0].set_title('ReLU: Hidden Layers\n(Default Choice)', fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].axvline(0, color='black', linewidth=0.5)
axes[0].set_xlabel('Input')
axes[0].set_ylabel('Output')

# Sigmoid
axes[1].plot(x.numpy(), y_sigmoid.numpy(), linewidth=2, color='green')
axes[1].set_title('Sigmoid: Binary\nClassification Output', fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.5)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].axvline(0, color='black', linewidth=0.5)
axes[1].set_xlabel('Input')
axes[1].set_ylabel('Output')

# Tanh
axes[2].plot(x.numpy(), y_tanh.numpy(), linewidth=2, color='orange')
axes[2].set_title('Tanh: Zero-Centered\nOutputs', fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].axvline(0, color='black', linewidth=0.5)
axes[2].set_xlabel('Input')
axes[2].set_ylabel('Output')

plt.tight_layout()
plt.show()

print("\nDECISION GUIDE:")
print("  Hidden Layers → ReLU")
print("  Binary Classification Output → Sigmoid")
print("  Multi-class Classification Output → Softmax")
print("  Need Zero-Centered → Tanh")

### 🔍 Exercise 2.1: Test Different Activations

Apply different activations to the same input and compare outputs.

In [ ]:
# TODO: Test activations on this input
test_input = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

relu_output = None      # Apply ReLU
sigmoid_output = None   # Apply Sigmoid
tanh_output = None      # Apply Tanh

print(f"Input: {test_input}")
print(f"ReLU: {relu_output}")
print(f"Sigmoid: {sigmoid_output}")
print(f"Tanh: {tanh_output}")

---
# Part 3: Loss Functions

## 3.1 Mean Squared Error (MSE) for Regression

In [ ]:
# MSE Loss for regression
criterion_mse = nn.MSELoss()

# Example: House price prediction
predictions = torch.tensor([300000., 250000., 450000., 350000.])
actual = torch.tensor([310000., 240000., 460000., 330000.])

mse_loss = criterion_mse(predictions, actual)

print("Regression Example (House Prices):")
print(f"  Predictions: {predictions}")
print(f"  Actual: {actual}")
print(f"  MSE Loss: {mse_loss.item():.2f}")

# Visualize errors
errors = (predictions - actual).abs()

plt.figure(figsize=(10, 6))
x_pos = np.arange(len(predictions))
plt.bar(x_pos, errors.numpy(), color=['red', 'blue', 'green', 'orange'], alpha=0.7)
plt.xlabel('House', fontsize=12)
plt.ylabel('Absolute Error ($)', fontsize=12)
plt.title('Prediction Errors (MSE penalizes squared errors)', fontsize=14, fontweight='bold')
plt.xticks(x_pos, [f'House {i+1}' for i in range(len(predictions))])
plt.grid(True, alpha=0.3, axis='y')
plt.show()

## 3.2 Cross-Entropy Loss for Classification

In [ ]:
# Cross-Entropy Loss for multi-class classification
criterion_ce = nn.CrossEntropyLoss()

# Example: 3 classes, 4 samples
logits = torch.tensor([
    [2.0, 1.0, 0.1],  # Sample 1: predicts class 0 (highest logit)
    [0.5, 2.5, 0.3],  # Sample 2: predicts class 1
    [1.0, 0.5, 3.0],  # Sample 3: predicts class 2
    [3.0, 0.2, 0.5]   # Sample 4: predicts class 0
])

labels = torch.tensor([0, 1, 2, 0])  # True class indices

ce_loss = criterion_ce(logits, labels)

print("Classification Example:")
print(f"  Logits shape: {logits.shape}")
print(f"  True labels: {labels}")
print(f"  Cross-Entropy Loss: {ce_loss.item():.4f}")

# Convert logits to probabilities
probs = F.softmax(logits, dim=1)

print("\nPredicted Probabilities:")
print(probs)

# Visualize for one sample
sample_idx = 0
plt.figure(figsize=(8, 5))
plt.bar(['Class 0', 'Class 1', 'Class 2'], probs[sample_idx].numpy(), alpha=0.7)
plt.axhline(probs[sample_idx, labels[sample_idx]].item(), color='red',
           linestyle='--', label=f'True class prob: {probs[sample_idx, labels[sample_idx]]:.3f}')
plt.ylabel('Probability', fontsize=12)
plt.title(f'Sample {sample_idx+1}: Predicted Class Probabilities', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.ylim(0, 1)
plt.show()

### 🔍 Exercise 3.1: Choose the Right Loss

For each scenario, select MSE or Cross-Entropy.

In [ ]:
scenarios = [
    "Predicting tomorrow's temperature",
    "Classifying email as spam or not spam",
    "Predicting stock price",
    "Recognizing handwritten digits (0-9)",
    "Estimating person's age from photo"
]

# TODO: For each scenario, print "MSE" or "Cross-Entropy"
answers = []  # Fill this list

for scenario, answer in zip(scenarios, answers):
    print(f"{scenario}: {answer}")

---
# Part 4: Forward Pass in PyTorch

## 4.1 The nn.Module Pattern

In [ ]:
class SimpleNet(nn.Module):
    """
    Simple 3-layer network using nn.Module.

    This is the standard PyTorch pattern!
    """
    def __init__(self, input_size, hidden_size, num_classes):
        super(SimpleNet, self).__init__()

        # Define layers in __init__
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # Define forward pass
        x = F.relu(self.fc1(x))
        x = self.fc2(x)  # No activation (use with CrossEntropyLoss)
        return x

# Create model
model = SimpleNet(input_size=784, hidden_size=128, num_classes=10)

print("Model Architecture:")
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

## 4.2 Forward Pass Example

In [ ]:
# Create sample input (batch of 32 MNIST-like images)
batch_size = 32
input_data = torch.randn(batch_size, 784)

print(f"Input shape: {input_data.shape}")

# Forward pass
with torch.no_grad():  # No gradient computation (inference)
    output = model(input_data)

print(f"Output shape: {output.shape}")
print(f"\nFirst sample logits: {output[0]}")

# Convert to probabilities
probs = F.softmax(output, dim=1)
predicted_classes = torch.argmax(output, dim=1)

print(f"\nFirst sample probabilities: {probs[0]}")
print(f"Predicted class: {predicted_classes[0].item()}")
print(f"\nBatch predictions: {predicted_classes[:10]}")

## 4.3 Step-by-Step Forward Pass Walkthrough

In [ ]:
class VerboseNetwork(nn.Module):
    """
    Network that prints shapes at each step.
    """
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x, verbose=False):
        if verbose:
            print(f"Input: {x.shape}")

        x = self.fc1(x)
        if verbose:
            print(f"After fc1: {x.shape}")

        x = F.relu(x)
        if verbose:
            print(f"After ReLU: {x.shape}, zeros: {(x == 0).sum().item()}")

        x = self.fc2(x)
        if verbose:
            print(f"After fc2: {x.shape}")

        x = F.relu(x)
        if verbose:
            print(f"After ReLU: {x.shape}")

        x = self.fc3(x)
        if verbose:
            print(f"Output: {x.shape}")

        return x

verbose_net = VerboseNetwork()
test_input = torch.randn(1, 784)

print("=" * 50)
print("FORWARD PASS WALKTHROUGH")
print("=" * 50)
output = verbose_net(test_input, verbose=True)

## 4.4 Building Your First Complete Network

In [ ]:
class MNISTClassifier(nn.Module):
    """
    Complete network for MNIST digit classification.
    """
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)

        x = F.relu(self.fc2(x))
        x = self.dropout(x)

        x = self.fc3(x)
        return x

mnist_model = MNISTClassifier()

print("MNIST Classifier:")
print(mnist_model)
print(f"\nTotal parameters: {sum(p.numel() for p in mnist_model.parameters()):,}")

---
# Part 5: Complete Training Example

## 5.1 Prepare Data

In [ ]:
# Create synthetic dataset
torch.manual_seed(42)
X_train = torch.randn(1000, 784)
y_train = torch.randint(0, 10, (1000,))

# Create DataLoader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(f"Training samples: {len(X_train)}")
print(f"Batch size: 32")
print(f"Number of batches: {len(train_loader)}")

## 5.2 Training Loop

In [ ]:
# Create fresh model
model = MNISTClassifier()

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training
num_epochs = 10
losses = []

print("Training...")
print("=" * 50)

for epoch in range(num_epochs):
    epoch_loss = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        # Forward pass
        output = model(data)
        loss = criterion(output, target)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)

    if (epoch + 1) % 2 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

print("=" * 50)
print("Training complete!")

## 5.3 Visualize Training

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(losses, linewidth=2, marker='o')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Loss Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Final loss: {losses[-1]:.4f}")

## 5.4 Make Predictions

In [ ]:
# Inference mode
model.eval()

with torch.no_grad():
    # Test on a single sample
    test_sample = torch.randn(1, 784)
    output = model(test_sample)
    probs = F.softmax(output, dim=1)
    predicted = torch.argmax(output, dim=1)

print("Single Prediction:")
print(f"  Predicted class: {predicted.item()}")
print("\n  Class probabilities:")
for i, p in enumerate(probs[0]):
    bar = '█' * int(p * 50)
    print(f"    Class {i}: {p:.4f} {bar}")

---
# Summary and Key Takeaways

## What We Learned

### 1. Perceptron to Multi-Layer
- Perceptron: Weighted sum + activation
- Single perceptron = linear boundary only
- Multi-layer networks learn hierarchies
- Depth matters only with non-linear activations

### 2. Activation Functions
- **ReLU**: Default for hidden layers
- **Sigmoid**: Binary classification output
- **Tanh**: Zero-centered alternative
- **Softmax**: Multi-class classification output
- Without activation: depth is useless!

### 3. Loss Functions
- **MSE**: Regression (continuous values)
- **CrossEntropyLoss**: Multi-class classification
- **BCELoss**: Binary classification
- Choice depends on task, not architecture

### 4. PyTorch Implementation
- Inherit from `nn.Module`
- Define layers in `__init__`
- Define forward pass in `forward()`
- Automatic gradient computation

## Decision Guides

**Activation Choice:**
```python
# Hidden layers
x = F.relu(self.fc1(x))  # Default: ReLU

# Output layer
# Binary classification
x = torch.sigmoid(self.fc_out(x))
# Multi-class classification  
x = self.fc_out(x)  # No activation, use with CrossEntropyLoss
```

**Loss Choice:**
```python
# Regression
criterion = nn.MSELoss()

# Binary classification
criterion = nn.BCELoss()

# Multi-class classification
criterion = nn.CrossEntropyLoss()
```

**Parameter Count:**
```python
# For Linear(in, out)
params = (in × out) + out
       = weights    + biases
```

## Next Steps

1. Train on real datasets (MNIST, CIFAR-10)
2. Add regularization (dropout, weight decay)
3. Learn backpropagation and gradient descent
4. Explore convolutional and recurrent networks

---

**Congratulations!** You now understand neural network fundamentals and can build networks in PyTorch! 🎉